# Task 4: In-Memory HNSW Vector Indexing from Scratch


TASK to Be done:
- store vectors in memory
- assign each point a random maximum layer
- connect nearby nodes
- search by greedy graph traversal


In [1]:
import math
import random
import numpy as np
import networkx as nx

np.random.seed(42)
random.seed(42)


In [2]:
def cosine_similarity(a, b):
    numerator = np.dot(a, b)
    denominator = np.linalg.norm(a) * np.linalg.norm(b) + 1e-12
    return numerator / denominator


class SimpleHNSW:
    def __init__(self, m=3, max_layers=3):
        self.m = m
        self.max_layers = max_layers
        self.vectors = []
        self.layers = []
        self.graphs = [nx.Graph() for _ in range(max_layers)]

    def random_level(self):
        level = 0
        while level < self.max_layers - 1 and random.random() < 0.5:
            level += 1
        return level

    def add_vector(self, vector):
        node_id = len(self.vectors)
        self.vectors.append(vector)
        max_level = self.random_level()
        self.layers.append(max_level)

        for layer in range(max_level + 1):
            self.graphs[layer].add_node(node_id)

            # Connect the new node to the top-m most similar existing nodes.
            candidates = []
            for other_id in self.graphs[layer].nodes:
                if other_id == node_id:
                    continue
                score = cosine_similarity(vector, self.vectors[other_id])
                candidates.append((score, other_id))

            candidates.sort(reverse=True)
            for _, other_id in candidates[: self.m]:
                self.graphs[layer].add_edge(node_id, other_id)

    def search(self, query_vector, top_k=3):
        if not self.vectors:
            return []

        # Start from node 0 for simplicity.
        current = 0

        # Greedy walk from the highest layer down.
        for layer in reversed(range(self.max_layers)):
            if current not in self.graphs[layer]:
                continue

            improved = True
            while improved:
                improved = False
                current_score = cosine_similarity(query_vector, self.vectors[current])
                for neighbor in self.graphs[layer].neighbors(current):
                    neighbor_score = cosine_similarity(query_vector, self.vectors[neighbor])
                    if neighbor_score > current_score:
                        current = neighbor
                        improved = True
                        break

        # Final ranking on layer 0 neighbors + current node.
        candidate_ids = set([current])
        candidate_ids.update(self.graphs[0].neighbors(current))
        scored = [(cosine_similarity(query_vector, self.vectors[idx]), idx) for idx in candidate_ids]
        scored.sort(reverse=True)
        return scored[:top_k]


In [3]:
index = SimpleHNSW(m=3, max_layers=4)

# Create a small toy dataset.
data = np.random.randn(15, 8).astype(np.float32)
for vector in data:
    index.add_vector(vector)

query = np.random.randn(8).astype(np.float32)
results = index.search(query, top_k=5)

print("Top matches:")
for score, node_id in results:
    print(f"Node {node_id}, cosine similarity = {score:.4f}")


Top matches:
Node 13, cosine similarity = 0.4036
Node 4, cosine similarity = 0.0826
Node 7, cosine similarity = -0.0266
Node 1, cosine similarity = -0.2972
